# Risk explanation & recommendations per loan

### Step 1: Risk Agent Processing

Load high-risk loans and apply agent logic to generate a risk score, 
identify risk drivers and assign a recommendation for each loan.

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# ── Load Gold Features ─────────────────────────────────────────────
df_spark = spark.read.table("credit_risk_assessment.gold.loan_features")

# Cast columns in Spark
feature_cols = ['debt_to_equity_ratio', 'loan_to_income_ratio',
                'days_past_due', 'credit_score', 'interest_rate',
                'revenue_decline_flag', 'low_credit_score_flag']

for c in feature_cols:
    df_spark = df_spark.withColumn(c, F.col(c).cast("double"))
df_spark = df_spark.withColumn("high_dpd_flag", F.col("high_dpd_flag").cast("int"))

# ── Filter High Risk Cases ─────────────────────────────────────────
risk_cases = df_spark.filter(F.col("high_dpd_flag") == 1)
print("High risk loan count:", risk_cases.count())

# ── Collect as plain Python rows ───────────────────────────────────
rows = risk_cases.select(
    "loan_id",
    "days_past_due",
    "debt_to_equity_ratio",
    "loan_to_income_ratio",
    "credit_score",
    "interest_rate",
    "revenue_decline_flag",
    "low_credit_score_flag",
    "high_dpd_flag"
).limit(20).collect()

# ── Agent Logic ────────────────────────────────────────────────────
def explain_risk(row):
    reasons = []
    if float(row["days_past_due"]) > 60:
        reasons.append("Payment Delays")
    if float(row["debt_to_equity_ratio"]) > 3:
        reasons.append("High Leverage")
    if float(row["loan_to_income_ratio"]) > 5:
        reasons.append("High Loan To Income")
    if float(row["credit_score"]) < 550:
        reasons.append("Low Credit Score")
    if float(row["interest_rate"]) > 15:
        reasons.append("High Interest Rate")
    if int(row["revenue_decline_flag"]) == 1:
        reasons.append("Revenue Decline")
    return reasons

def risk_score(row):
    score = 0
    if float(row["days_past_due"]) > 60:       score += 35
    if float(row["debt_to_equity_ratio"]) > 3:  score += 20
    if float(row["loan_to_income_ratio"]) > 5:  score += 15
    if float(row["credit_score"]) < 550:        score += 15
    if float(row["interest_rate"]) > 15:        score += 10
    if int(row["revenue_decline_flag"]) == 1:   score += 5
    return score

def recommendation(score, reasons):
    if score >= 50:
        return "REJECT - Immediate review required"
    elif score >= 30:
        return "REVIEW  - Flag for credit committee"
    else:
        return "MONITOR - Schedule follow up"

# ── Generate Report ────────────────────────────────────────────────
print("\n" + "="*85)
print(f"  {'LOAN_ID':<12} {'RISK_SCORE':<12} {'RISK_DRIVERS':<35} {'RECOMMENDATION'}")
print("="*85)

for row in rows:
    reasons   = explain_risk(row)
    score     = risk_score(row)
    rec       = recommendation(score, reasons)
    drivers   = ", ".join(reasons) if reasons else "None"
    loan_id   = str(row["loan_id"])

    print(f"  {loan_id:<12} {score:<12} {drivers:<35} {rec}")

print("="*85)
print(f"\n  Total high risk loans analysed: {len(rows)}")
print("="*85)

High risk loan count: 286



  LOAN_ID      RISK_SCORE   RISK_DRIVERS                        RECOMMENDATION
  44           85           Payment Delays, High Leverage, High Loan To Income, Low Credit Score REJECT - Immediate review required
  967          35           Payment Delays                      REVIEW  - Flag for credit committee
  398          55           Payment Delays, Low Credit Score, Revenue Decline REJECT - Immediate review required
  1587         50           Payment Delays, Low Credit Score    REJECT - Immediate review required
  374          70           Payment Delays, High Leverage, High Loan To Income REJECT - Immediate review required
  498          55           Payment Delays, High Leverage       REJECT - Immediate review required
  1206         35           Payment Delays                      REVIEW  - Flag for credit committee
  1706         35           Payment Delays                      REVIEW  - Flag for credit committee
  188          35           Payment Delays                     

### Step 2: Save Agent Output to Gold Layer

In [46]:
# save to Gold 
results = []

for row in rows:
    reasons = explain_risk(row)
    score   = risk_score(row)
    rec     = recommendation(score, reasons)

    results.append((
        str(row["loan_id"]),
        int(score),
        ", ".join(reasons) if reasons else "None",
        rec
    ))

# Create Spark DataFrame from plain Python list
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("loan_id",          StringType(),  True),
    StructField("risk_score",       IntegerType(), True),
    StructField("risk_drivers",     StringType(),  True),
    StructField("recommendation",   StringType(),  True),
])

results_df = spark.createDataFrame(results, schema=schema)

# Preview
results_df.show(10, truncate=False)

# Save to Gold
results_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("credit_risk_assessment.gold.loan_risk_report")



+-------+----------+--------------------------------------------------------------------+-----------------------------------+
|loan_id|risk_score|risk_drivers                                                        |recommendation                     |
+-------+----------+--------------------------------------------------------------------+-----------------------------------+
|44     |85        |Payment Delays, High Leverage, High Loan To Income, Low Credit Score|REJECT - Immediate review required |
|967    |35        |Payment Delays                                                      |REVIEW  - Flag for credit committee|
|398    |55        |Payment Delays, Low Credit Score, Revenue Decline                   |REJECT - Immediate review required |
|1587   |50        |Payment Delays, Low Credit Score                                    |REJECT - Immediate review required |
|374    |70        |Payment Delays, High Leverage, High Loan To Income                  |REJECT - Immediate review req

## Step 3 Generating loan master table with consolidated loan records to be utilized for analytics

In [47]:
from pyspark.sql import functions as F

# Load both gold tables
loan_features    = spark.read.table("credit_risk_assessment.gold.loan_features")
loan_risk_report = spark.read.table("credit_risk_assessment.gold.loan_risk_report")

# Cast loan_id to same type on both sides before join
loan_features    = loan_features.withColumn("loan_id", F.col("loan_id").cast("integer"))
loan_risk_report = loan_risk_report.withColumn("loan_id", F.col("loan_id").cast("integer"))

# Join — keep all rows from loan_risk_report (right join)
loan_master = loan_risk_report.join(loan_features, on="loan_id", how="left")

print("loan_risk_report rows :", loan_risk_report.count())
print("loan_features rows    :", loan_features.count())
print("loan_master rows      :", loan_master.count())
loan_master.show(5, truncate=False)

loan_risk_report rows : 20


loan_features rows    : 10000


loan_master rows      : 106


+-------+----------+--------------+-----------------------------------+-----------+-----------+---------+-------------+----------------+----------------+----------+--------------+-------------+--------------+----------+--------+--------+---------------+-------------+---+-------------+-----------------+------------+-------------+--------------------+--------------------+---------------------+--------------------+
|loan_id|risk_score|risk_drivers  |recommendation                     |customer_id|loan_amount|loan_type|interest_rate|loan_term_months|collateral_value|payment_id|payment_amount|days_past_due|payment_status|total_debt|equity  |revenue |revenue_decline|customer_name|age|annual_income|employment_status|credit_score|high_dpd_flag|debt_to_equity_ratio|loan_to_income_ratio|low_credit_score_flag|revenue_decline_flag|
+-------+----------+--------------+-----------------------------------+-----------+-----------+---------+-------------+----------------+----------------+----------+----

## Step 4 : Saving the tables to vector database for AI agents and downstream applications

In [ ]:
# save to vector database
loan_master.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ai_lakehouse.aidp.loan_summary")

loan_risk_report.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ai_lakehouse.aidp.loan_risk_report")

loan_features.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ai_lakehouse.aidp.loan_features")
